Georgia Institute of Technology

Earth and Atmospheric Sciences

Geophysics laboratory

Nathalie Chavarria

In [36]:
import os
import re
import pygmt
import numpy as np
from glob import glob
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

In [37]:
## variables 
minlon, maxlon = -87, -82.5
minlat, maxlat = 7.0, 11.2
# minlon, maxlon = -87, -81.5
# minlat, maxlat = 6.0, 12.2
topo_data = '@earth_relief_15s' #arcosecond global relief SRTM
events = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/processing/bigger_6.txt" # events from Costa Rica
#stations = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/CR_GNSS/lista.txt"

# velocity data
vel_path = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/rates/CR_rates_ovsicori.txt"
bad_path = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/rates/CR_rates_bad.txt"
earth_path = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/rates/CR_rates_earthscope.txt"

# magscale = "mag.dat"

#import plates
panama = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/pana_digi.dat"
nazca = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/nazca_digi.dat"
coco = "/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/mapping/trench.ll"
lonpana,latpana = np.genfromtxt(panama, usecols=(1, 0), unpack=True,)
lonaz,latnaz = np.genfromtxt(nazca, usecols=(1, 0), unpack=True,)
lonco,latco = np.genfromtxt(coco, usecols=(0, 1), unpack=True,)

In [38]:
# get data from velocity file 
df = pd.read_csv(vel_path, delim_whitespace=True, header=None, comment="#", names=["Stat","Lat", "Long", "Height","Nvel", "Evel", "Uvel", "Nerr", "Eerr", "Uerr", "Corr_NE", "Corr_NU", "Corr_EU", "Sdate", "Edate", "InstallYear"])
# define variables GNSS
lon = df['Long']
lat = df['Lat']
stat = df['Stat']

#dmin, dmax = df['InstallYear'].min(), df['InstallYear'].max()
dmin, dmax = 2000, 2025

#print(df)

df_earth = pd.read_csv(
    earth_path,
    delim_whitespace=True,
    header=None,
    comment="#",
    names=[
        "Stat", "Lat", "Long", "Height",
        "Nvel", "Evel", "Uvel", "Nvel-loc", "Evel-loc",
        "Nerr", "Eerr", "Uerr",
        "NEcor", "NUcor", "EUcor",
        "Sdate", "Edate", "InstallYear"
    ]
)

# define variables GNSS
lone = df_earth["Long"]
late = df_earth["Lat"]
state = df_earth["Stat"]

#print(df_earth.head())
print(lone[0], late[0], state[0])


-82.2563 9.3517 CN20


/var/folders/fm/zxb00jfj0zdcxft7sfv1gps00000gn/T/ipykernel_8223/2831294934.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(vel_path, delim_whitespace=True, header=None, comment="#", names=["Stat","Lat", "Long", "Height","Nvel", "Evel", "Uvel", "Nerr", "Eerr", "Uerr", "Corr_NE", "Corr_NU", "Corr_EU", "Sdate", "Edate", "InstallYear"])
/var/folders/fm/zxb00jfj0zdcxft7sfv1gps00000gn/T/ipykernel_8223/2831294934.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_earth = pd.read_csv(


In [39]:
## bad stations
df_bad = pd.read_csv(bad_path, delim_whitespace=True, header=None, comment="#", names=["Stat","Lat", "Long", "Height","Nvel", "Evel", "Uvel", "Nerr", "Eerr", "Uerr", "Corr_NE", "Corr_NU", "Corr_EU", "Sdate", "Edate", "InstallYear"])
# define variables GNSS
lon_bad = df_bad['Long']
lat_bad = df_bad['Lat']
stat_bad = df_bad['Stat']

/var/folders/fm/zxb00jfj0zdcxft7sfv1gps00000gn/T/ipykernel_8223/3596696921.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_bad = pd.read_csv(bad_path, delim_whitespace=True, header=None, comment="#", names=["Stat","Lat", "Long", "Height","Nvel", "Evel", "Uvel", "Nerr", "Eerr", "Uerr", "Corr_NE", "Corr_NU", "Corr_EU", "Sdate", "Edate", "InstallYear"])


In [40]:
# Start plotting
fig = pygmt.Figure()
pygmt.config(MAP_FRAME_TYPE="plain")
pygmt.config(FORMAT_GEO_MAP="ddd")
pygmt.config(PS_LINE_CAP="round")
### Main figure
fig.shift_origin(yshift="5c", )
with fig.subplot(nrows=1, ncols=1, figsize=("15c", "15c"), ):
    fig.coast(
        region=[minlon, maxlon, minlat, maxlat],
        shorelines=True,
        borders=["1/0.5,black"],
        frame=['WSne','a1'],
        projection="M20c",
    )
    # topography
    fig.grdimage(
        grid=topo_data,
        region=[minlon, maxlon, minlat, maxlat],
        cmap="gray",
        shading=True,
        transparency=70,
        projection="M20c",
    )
    # contour
    fig.grdcontour(
        grid=topo_data,
        levels=500,
        limit=[-8000,0],
        pen=0.3,
        projection="M20c",
    )
    #box for leyend , has to be below everything else
    fig.plot(x=[-83.58, -86.986, -86.986, -83.58], y=[7.012, 7.012, 7.31, 7.31,],  pen="1p,black", fill="white", transparency=30, projection="M20c",)
    fig.coast(map_scale=["-84.58/7.21/20/100+lKilometers/w0.5c", ],  shorelines='0.5p,black', borders=['1/0.5p,black'], projection="M20c", )
    
    # plot plates
    fig.plot(x=lonpana, y=latpana, pen="1p,darkred,-", projection="M20c", )
    fig.plot(x=lonaz, y=latnaz, pen="1p,darkred", projection="M20c", style="f1.2c/0.2c+l+t", fill="darkred",)
    fig.plot(x=lonco, y=latco, pen="1p,darkred", projection="M20c", style="f1.2c/0.2c+l+t", fill="darkred",)

    cmap = pygmt.makecpt(cmap='viridis', series=[dmin, dmax], reverse=True)
    
    ##plot the stations
    fig.plot(
        x=lon,
        y=lat,
        style="t0.4c",
        pen="0.5,black",
        fill=df['InstallYear'],
        #fill="black",
        cmap=True,
        #label=" GNSS Stations"
        projection="M20c",

    )
    #name of LAFE
    fig.text( x=-84.96028445, y=9.8070616343, text="LAFE", justify="ML", offset="0.2c/-0.3c", projection="M20c",)

    ##plot earthscope stations
    fig.plot(
        x=lone,
        y=late,
        style="t0.4c",
        pen="0.5,black",
        fill=df_earth['InstallYear'],
        #fill="black",
        cmap=True,
        #label=" GNSS Stations"
        projection="M20c",)
    
    ##plot the bad stations
    fig.plot(
        x=lon_bad,
        y=lat_bad,
        style="t0.4c",
        pen="0.5,black",
        fill="lightgrey",
        projection="M20c",
        transparency=30,

    )
    ## plot bad velocity vectors
    ### create dataframe for vectors
    df_vel_bad=df_bad[['Long', 'Lat', 'Evel', 'Nvel', 'Eerr', 'Nerr', 'Corr_NE', ]] #,'Stat'
    fig.velo(
        data=df_vel_bad,
        projection="M20c",
        spec="e0.1/0.95+f4p",
        #uncertaintyfill="lightblue1",
        pen="1p,grey",
        line=True,
        vector="0.3c+p0.3p+e+ggrey",) 

    ## plot velocity vectors
    ### create dataframe for vectors
    df_vel=df[['Long', 'Lat', 'Evel', 'Nvel', 'Eerr', 'Nerr', 'Corr_NE', ]] #,'Stat'
    fig.velo(
        data=df_vel,
        projection="M20c",
        spec="e0.1/0.95+f4p", ## 95 confidence interval 
        #uncertaintyfill="lightblue1",
        pen="1p,black",
        line=True,
        vector="0.4c+p0.3p+e+gred",)  
    
    ## plot velocity vectors earthscope
    ### create dataframe for vectors
    df_vel2=df_earth[['Long', 'Lat', 'Evel-loc', 'Nvel-loc', 'Eerr','Nerr', 'NEcor', ]] #,'Stat'
    fig.velo(
        data=df_vel2,
        projection="M20c",
        spec="e0.1/0.95+f4p", ## 95 confidence interval 
        #uncertaintyfill="lightblue1",
        pen="1p,black",
        line=True,
        vector="0.4c+p0.3p+e+gmagenta",)  
    ## plot reference vector 3 cm 
    #fig.plot(x=[-83.48, -83.99, -83.99, -83.48], y=[7.14, 7.14, 7.008, 7.008,],  pen="1p,black", fill="white", transparency=30, projection="M20c",)
    df_ref = pd.DataFrame(data={"Long": [-84.05], "Lat": [7.21], "Evel": [20.0], "Nvel": [0.0], "Eerr": [0.0], "Nerr": [0.0], "Corr_NE": [0.0],})
    fig.velo(
        data=df_ref,
        projection="M20c",
        spec="e0.1/0.95+f4p", ## 95 confidence interval 
        #uncertaintyfill="lightblue1",
        pen="1p,black",
        line=True,
        vector="0.4c+p0.3p+e+gred",)
    
    # vectores

    fig.plot(x=[-85.5], y=[8.3], style="v0.6c+e", direction=([65], [1.5]), pen="2p", fill="black", projection="M20c",)
    fig.text(text=["88mm/yr"], x=[-85.55], y=[8.4], angle=65, projection="M20c",)

    fig.plot(x=[-83.8], y=[7.5], style="v0.6c+e", direction=([65], [1.5]), pen="2p", fill="black", projection="M20c",)
    fig.text(text=["91mm/yr"], x=[-83.85], y=[7.6], angle=65, projection="M20c",)

    fig.plot(x=[-86.65], y=[9.4], style="v0.6c+e", direction=([65], [1.5]), pen="2p", fill="black", projection="M20c",)
    fig.text(text=["85mm/yr"], x=[-86.7], y=[9.5], angle=65, projection="M20c",)
    
    
    fig.text( x=-83.99, y=7.12, text="20 mm/yr ", justify="ML", offset="0.0c/-0c", projection="M20c",) 
    
    fig.image(imagefile="/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/results_maps/Lafe_Raw_GNSS_Time_Series.png",
    position="JBM+o0.01c/-12c+w20c/4c+o0.1c", projection="M20c")

    # # additionals
    fig.colorbar(frame=['xaf+lStation year of installation', 'yaf'], position="jBL+o0.41c/0.55c+h+w8c/0.24c",  ) # jBL+o-0.35c/-8.9c+w8c/0.24c" box="+gwhite@20+p0.8p,black",


    fig.savefig("/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/results_maps/velocity_vectors_raw.png", crop=True, dpi=700)
